# Outlier Detection: A Comprehensive Guide
## From Statistical Foundations to Modern Machine Learning Approaches

---

### Table of Contents

1. **Introduction to Outliers** — Definitions, types, and motivation
2. **Taxonomy of Detection Methods** — A classification framework
3. **Statistical Methods** — Z-Score, IQR, and Grubbs' Test
4. **Isolation Forest** — Random partitioning for anomaly isolation
5. **Local Outlier Factor (LOF)** — Density-based local anomaly detection
6. **PCA-Based Outlier Detection** — Reconstruction error in reduced dimensions
7. **One-Class SVM** — Kernel-based boundary learning in feature space
8. **DBSCAN** — Density-based spatial clustering for noise detection
9. **Elliptic Envelope** — Robust covariance and Mahalanobis distance
10. **Autoencoders** — Neural network reconstruction-based anomaly detection
11. **Comparative Analysis** — Choosing the right method

---

> *"An outlier is an observation which deviates so much from the other observations as to arouse suspicions that it was generated by a different mechanism."*
> — D.M. Hawkins, *Identification of Outliers* (1980)

## 1. Introduction to Outliers

### 1.1 What is an Outlier?

An **outlier** (also called an **anomaly** or **novelty**) is a data point that significantly deviates from the expected pattern of the majority of data. Formally, given a dataset $$\mathcal{D} = \{x_1, x_2, \ldots, x_n\}$$ where each $$x_i \in \mathbb{R}^d$$, an outlier is a point $$x_j$$ for which:

$$p(x_j \mid \theta) < \tau$$

where $$\theta$$ parameterizes the data-generating distribution and $$\tau$$ is a decision threshold. In practice, the true distribution is unknown, and the entire field of outlier detection revolves around estimating this probability — or a suitable proxy — from data.

### 1.2 Why Outlier Detection Matters

| Domain | Application | Impact |
| --- | --- | --- |
| Finance | Credit card fraud detection | Billions in fraud prevented annually |
| Healthcare | Rare disease identification, adverse drug reactions | Early diagnosis saves lives |
| Manufacturing | Defect detection in production lines | Reduced waste and product recalls |
| Cybersecurity | Network intrusion detection | Prevention of data breaches |
| E-commerce | Bot detection, fake reviews, price manipulation | Marketplace integrity |
| IoT / Predictive Maintenance | Sensor failure detection | Downtime prevention |

### 1.3 Types of Outliers

**Point (Global) Outliers** — A single observation that lies far from the rest of the data in the global feature space. *Example*: A credit card transaction of \$50,000 when the cardholder's average is \$50.

**Contextual (Conditional) Outliers** — An observation that is anomalous only within a specific context defined by contextual attributes. *Example*: A temperature of 35°C is normal in July but anomalous in December for the same city.

**Collective Outliers** — A group of observations that are jointly anomalous, even though each individual point may appear normal. *Example*: A sudden burst of identical network packets from one source (possible DDoS attack).

### 1.4 Key Challenges

* **No labeled anomalies**: Most real-world settings are unsupervised — we rarely have ground truth
* **Extreme class imbalance**: Outliers constitute <1% (often <0.01%) of data
* **Curse of dimensionality**: Distance metrics lose discriminative power in high-dimensional spaces
* **Concept drift**: The definition of "normal" evolves over time
* **Masking and swamping**: Multiple outliers can mask each other, or inliers can be swamped by nearby outliers
* **Subjectivity**: The boundary between normal and anomalous is inherently fuzzy

## 2. Taxonomy of Outlier Detection Methods

### 2.1 By Supervision Level

| Paradigm | Description | When to Use |
| --- | --- | --- |
| **Supervised** | Trained on labeled normal + anomaly examples | Sufficient labeled anomalies available (rare in practice) |
| **Semi-supervised** | Trained only on "normal" data; anything deviating is flagged | Clean normal data available, no anomaly labels |
| **Unsupervised** | No labels at all; assumes anomalies are rare & different | Most common real-world setting |

### 2.2 By Algorithmic Family

**Statistical / Probabilistic** — Fit a statistical model (Gaussian, mixture, etc.) and flag low-probability points. *Methods*: Z-Score, Grubbs' Test, IQR, Elliptic Envelope.

**Proximity-Based (Distance / Density)** — Use distances or local density to identify isolated points. *Methods*: KNN-based, LOF, LOCI, COF.

**Linear Model-Based** — Project data onto principal subspaces; anomalies have high reconstruction error. *Methods*: PCA, Robust PCA, Autoencoder (linear case).

**Ensemble / Tree-Based** — Combine multiple weak detectors or exploit tree partitioning. *Methods*: Isolation Forest, Random Cut Forest, Feature Bagging.

**Boundary-Based** — Learn a tight boundary around normal data in kernel space. *Methods*: One-Class SVM, SVDD.

**Clustering-Based** — Cluster the data; points not belonging to any cluster (or in small clusters) are anomalies. *Methods*: DBSCAN, k-Means residual.

**Deep Learning** — Use neural networks to learn complex representations of normality. *Methods*: Autoencoders, VAE, GAN-based, Deep SVDD.

### 2.3 A Visual Roadmap

```
                          Outlier Detection
                                |
            ┌───────────────────┼───────────────────┐
        Statistical         Model-Based          Data-Driven
            |                   |                    |
     ┌──────┴──────┐     ┌─────┴─────┐       ┌──────┴──────┐
   Z-Score   IQR   Grubbs PCA  OCSVM       IF   LOF  DBSCAN
                          |                         |
                    Elliptic Envelope          Autoencoders
```

In the sections that follow, we cover each major method with its **intuition**, **mathematical theory**, **optimization derivation**, **implementation**, and **industrial application**.

In [0]:
# ============================================================
# Environment Setup — Core Libraries for Outlier Detection
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: models & utilities
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score
)

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'legend.fontsize': 10,
})
sns.set_style("whitegrid")

print("✓ All libraries loaded successfully.")

In [0]:
# ============================================================
# Generate a Reusable Synthetic Dataset with Known Outliers
# ============================================================
# We create a controlled 2D dataset so every algorithm can be
# visually compared on the same ground truth.

np.random.seed(42)

# --- Normal data: two Gaussian clusters ---
n_normal = 300
cluster_1 = np.random.randn(n_normal // 2, 2) * 0.8 + np.array([2, 2])
cluster_2 = np.random.randn(n_normal // 2, 2) * 0.6 + np.array([-2, -1])
X_normal = np.vstack([cluster_1, cluster_2])

# --- Outliers: uniformly scattered in a wider region ---
n_outliers = 30
X_outliers = np.random.uniform(low=-8, high=8, size=(n_outliers, 2))

# --- Combined dataset ---
X = np.vstack([X_normal, X_outliers])
y_true = np.array([1] * n_normal + [-1] * n_outliers)  # 1 = inlier, -1 = outlier

# --- Visualization ---
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_normal[:, 0], X_normal[:, 1], c='steelblue', s=20, alpha=0.6, label='Normal')
ax.scatter(X_outliers[:, 0], X_outliers[:, 1], c='red', s=50, marker='x', label='Outlier')
ax.set_title('Synthetic Dataset — Ground Truth')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Dataset: {X.shape[0]} samples ({n_normal} normal, {n_outliers} outliers) in {X.shape[1]} dimensions")

## 3. Statistical Methods for Outlier Detection

Statistical methods are the oldest and most interpretable family of outlier detectors. They assume the data follows a known distribution and flag points that are improbable under that model.

### 3.1 Z-Score Method

**Intuition**: If the data is approximately Gaussian, most points lie within a few standard deviations of the mean. Points far from the mean are unlikely.

**Definition**: For a univariate dataset, the Z-score of observation $$x_i$$ is:

$$z_i = \frac{x_i - \mu}{\sigma}$$

where $$\mu = \frac{1}{n}\sum_{i=1}^{n} x_i$$ is the sample mean and $$\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \mu)^2}$$ is the sample standard deviation.

A point is flagged as an outlier if $$|z_i| > \alpha$$, where $$\alpha$$ is typically 2.5 or 3.

**For multivariate data**, the Z-score generalizes to the **Mahalanobis distance** (covered in Section 9).

**Limitations**: Assumes Gaussianity; sensitive to the very outliers it aims to detect (the mean and std are pulled toward outliers — the *masking effect*).

---

### 3.2 IQR (Interquartile Range) Method

**Intuition**: Uses the median and quartiles instead of the mean and standard deviation, making it **robust** to outliers.

**Definition**: Let $$Q_1$$ and $$Q_3$$ be the 25th and 75th percentiles. The IQR is:

$$\text{IQR} = Q_3 - Q_1$$

A point $$x_i$$ is an outlier if:

$$x_i < Q_1 - k \cdot \text{IQR} \quad \text{or} \quad x_i > Q_3 + k \cdot \text{IQR}$$

The standard multiplier is $$k = 1.5$$ ("mild outlier") or $$k = 3.0$$ ("extreme outlier"). This is the foundation of the classic **box plot**.

**Why $$k = 1.5$$?** For a perfect Gaussian, $$Q_1 \approx \mu - 0.6745\sigma$$ and $$Q_3 \approx \mu + 0.6745\sigma$$, so $$\text{IQR} \approx 1.349\sigma$$. The fences at $$k=1.5$$ correspond roughly to $$\mu \pm 2.7\sigma$$, capturing about 99.3% of the data.

---

### 3.3 Grubbs' Test

**Intuition**: A formal hypothesis test specifically designed to detect a single outlier in a univariate Gaussian sample.

**Hypothesis**:
* $$H_0$$: There are no outliers in the dataset
* $$H_1$$: There is exactly one outlier

**Test Statistic**:

$$G = \frac{\max_{i} |x_i - \bar{x}|}{s}$$

where $$\bar{x}$$ is the sample mean and $$s$$ is the sample standard deviation.

**Critical value**: Reject $$H_0$$ at significance level $$\alpha$$ if:

$$G > \frac{n-1}{\sqrt{n}} \sqrt{\frac{t^2_{\alpha/(2n),\, n-2}}{n - 2 + t^2_{\alpha/(2n),\, n-2}}}$$

where $$t_{\alpha/(2n),\, n-2}$$ is the critical value of the $$t$$-distribution with $$n-2$$ degrees of freedom.

**Limitation**: Tests for only one outlier at a time. For multiple outliers, apply iteratively (remove detected outlier, re-test) — but this is susceptible to masking.

---

**Industrial Example — Quality Control in Pharmaceuticals**: A pharmaceutical company measures tablet weights on a production line. Tablets are expected to weigh $$500 \pm 5$$ mg (Gaussian). The Z-score and IQR methods flag tablets outside tolerance for rejection. Grubbs' test is applied to each batch to formally certify it is outlier-free before release.

In [0]:
# ============================================================
# Statistical Methods: Z-Score and IQR on the Synthetic Dataset
# ============================================================

# We apply both methods to each feature independently, then combine.

def zscore_outliers(X, threshold=3.0):
    """Flag outliers using the Z-score method (per feature)."""
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0, ddof=1)
    z_scores = np.abs((X - mean) / std)
    # A point is an outlier if ANY feature exceeds threshold
    return np.where(np.any(z_scores > threshold, axis=1), -1, 1)

def iqr_outliers(X, k=1.5):
    """Flag outliers using the IQR method (per feature)."""
    Q1 = np.percentile(X, 25, axis=0)
    Q3 = np.percentile(X, 75, axis=0)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    mask = np.any((X < lower) | (X > upper), axis=1)
    return np.where(mask, -1, 1)

y_zscore = zscore_outliers(X, threshold=3.0)
y_iqr = iqr_outliers(X, k=1.5)

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, labels, title in zip(
    axes,
    [y_true, y_zscore, y_iqr],
    ['Ground Truth', 'Z-Score (|z| > 3)', 'IQR (k=1.5)']
):
    colors = ['red' if l == -1 else 'steelblue' for l in labels]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    n_detected = np.sum(labels == -1)
    ax.annotate(f'Detected: {n_detected}', xy=(0.02, 0.95),
                xycoords='axes fraction', fontsize=11, color='red')

plt.suptitle('Statistical Methods Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 4. Isolation Forest

### 4.1 Intuition

Isolation Forest (Liu et al., 2008) is built on a beautifully simple insight:

> **Anomalies are few and different; therefore, they are easier to isolate.**

Imagine repeatedly choosing a random feature and a random split value to partition the data. Normal points, being clustered together, require many splits before they are separated from their neighbors. Anomalies, being sparse and distant, are isolated in very few splits.

This is the opposite of most methods, which build a profile of "normal" and then measure deviation. Isolation Forest directly measures how easy it is to **isolate** a point.

### 4.2 The Isolation Tree (iTree)

An **iTree** is a binary tree built by:
1. Randomly selecting a feature $$q$$ from the $$d$$ available features
2. Randomly selecting a split value $$p$$ uniformly between the minimum and maximum of feature $$q$$ in the current subset
3. Recursively partitioning the data into left ($$x_q < p$$) and right ($$x_q \geq p$$) nodes
4. Stopping when either:
   * The node contains a single point (fully isolated)
   * The tree reaches a maximum height $$l = \lceil \log_2 \psi \rceil$$ where $$\psi$$ is the subsample size

The key quantity is the **path length** $$h(x)$$: the number of edges from the root to the leaf containing point $$x$$. Anomalies have **short** path lengths; normal points have **long** ones.

### 4.3 Why Subsampling?

Isolation Forest uses subsampling (typically $$\psi = 256$$) rather than the full dataset. This:
* **Reduces computational cost** from $$O(n^2)$$ to $$O(n \cdot \psi)$$
* **Eliminates swamping**: with fewer points, outliers are less likely to be masked
* **Eliminates masking**: dense regions don't dominate the tree structure
* **Acts as implicit regularization**: preventing overfitting to the exact data distribution

### 4.4 The Anomaly Score — Full Derivation

The anomaly score normalizes path lengths to a $$[0, 1]$$ scale. The key reference is the **average path length of unsuccessful search in a Binary Search Tree (BST)**.

#### Step 1: Expected Path Length in a BST

For a BST built from $$n$$ points, the average path length of an unsuccessful search is:

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}$$

where $$H(k) = \ln(k) + \gamma$$ is the $$k$$-th harmonic number and $$\gamma \approx 0.5772$$ is the Euler–Mascheroni constant.

**Why BST?** An iTree with purely random splits on uniformly distributed data behaves like a BST. So $$c(n)$$ serves as the expected path length for "average" data — our normalization baseline.

#### Step 2: Average Path Length Across the Forest

Given an ensemble of $$T$$ isolation trees, the average path length for a point $$x$$ is:

$$E[h(x)] = \frac{1}{T}\sum_{t=1}^{T} h_t(x)$$

where $$h_t(x)$$ is the path length of $$x$$ in tree $$t$$.

#### Step 3: The Anomaly Score

The anomaly score is defined as:

$$s(x, \psi) = 2^{-\frac{E[h(x)]}{c(\psi)}}$$

where $$\psi$$ is the subsample size used to build each tree.

**Interpreting the score**:

| Condition | Meaning | Score |
| --- | --- | --- |
| $$E[h(x)] \to 0$$ | Isolated immediately (anomaly) | $$s \to 2^0 = 1$$ |
| $$E[h(x)] = c(\psi)$$ | Average path length (ambiguous) | $$s = 2^{-1} = 0.5$$ |
| $$E[h(x)] \to \psi - 1$$ | Never isolated (very normal) | $$s \to 2^{-(\psi-1)/c(\psi)} \approx 0$$ |

Points with $$s > 0.5$$ are increasingly likely to be anomalies. In sklearn, the convention is inverted: the `decision_function` returns $$-s$$ offset so that negative values indicate anomalies.

#### Step 4: Path Length Adjustment for External Nodes

When a leaf node at depth $$l$$ still contains more than one point (because we hit the height limit), we adjust:

$$h(x) = e + c(\text{Size})$$

where $$e$$ is the edge count to the leaf and $$\text{Size}$$ is the number of points in that leaf. This accounts for the expected additional depth if we had continued splitting.

### 4.5 Variants and Extensions

**Extended Isolation Forest (EIF)** — The original IF splits along axis-aligned hyperplanes, creating a bias toward detecting anomalies in axis-aligned directions. EIF (Hariri et al., 2019) uses **random hyperplanes with arbitrary slopes**: the split is defined by a random normal vector $$\mathbf{n}$$ and intercept $$p$$, partitioning on $$\mathbf{n} \cdot \mathbf{x} < p$$. This removes the axis-aligned bias and better handles correlations.

**SCiForest (Split-Criterion Isolation Forest)** — Uses a learned split criterion (e.g., kurtosis-based) rather than purely random splits, improving detection in the presence of irrelevant features.

**Functional Isolation Forest** — Extends IF to functional data (e.g., time series, curves) by splitting on random projections of the function space.

### 4.6 Hyperparameters

| Parameter | Meaning | Guidance |
| --- | --- | --- |
| `n_estimators` | Number of trees $$T$$ | 100–300 usually sufficient; diminishing returns beyond 300 |
| `max_samples` | Subsample size $$\psi$$ | 256 is the default and works well; increase only for very large, complex datasets |
| `contamination` | Expected fraction of outliers | Set to approximate anomaly rate if known; `'auto'` uses the score offset |
| `max_features` | Features sampled per tree | 1.0 (all features) by default; reduce for very high-dimensional data |

### 4.7 Complexity

* **Training**: $$O(T \cdot \psi \cdot \log \psi)$$ — very fast since $$\psi$$ is small
* **Scoring**: $$O(T \cdot \log \psi)$$ per point
* **Space**: $$O(T \cdot \psi)$$

This makes Isolation Forest one of the most scalable outlier detectors — easily handling millions of points.

## 6. PCA-Based Outlier Detection

### 6.1 Intuition

Principal Component Analysis (PCA) finds the directions of maximum variance in the data. Normal points lie close to the principal subspace; outliers, being structurally different, have large **reconstruction errors** when projected onto and back from this subspace.

Think of it as fitting a "hyperplane of normality." Points that don't fit — that can't be well-represented by the dominant patterns — are anomalies.

### 6.2 PCA Recap

Given centered data matrix $$\mathbf{X} \in \mathbb{R}^{n \times d}$$, PCA solves:

$$\max_{\mathbf{w}} \;\; \mathbf{w}^T \mathbf{S} \mathbf{w} \quad \text{subject to} \quad \|\mathbf{w}\| = 1$$

where $$\mathbf{S} = \frac{1}{n-1}\mathbf{X}^T\mathbf{X}$$ is the sample covariance matrix.

The solution is the eigenvector corresponding to the largest eigenvalue of $$\mathbf{S}$$. The top $$r$$ eigenvectors form the projection matrix $$\mathbf{W}_r = [\mathbf{w}_1, \ldots, \mathbf{w}_r] \in \mathbb{R}^{d \times r}$$.

### 6.3 The Reconstruction and the Error

For a point $$\mathbf{x}$$, the reconstruction via the top $$r$$ components is:

$$\hat{\mathbf{x}} = \mathbf{W}_r \mathbf{W}_r^T \mathbf{x}$$

The **reconstruction error** is:

$$\text{RE}(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 = \|\mathbf{x} - \mathbf{W}_r \mathbf{W}_r^T \mathbf{x}\|^2$$

This can be rewritten using the *minor* components (the $$d - r$$ eigenvectors with smallest eigenvalues):

$$\text{RE}(\mathbf{x}) = \sum_{j=r+1}^{d} (\mathbf{w}_j^T \mathbf{x})^2$$

Outliers have large projections onto the minor components — they carry variance in directions that normal data does not.

### 6.4 Outlier Score Derivation

#### The Optimization Problem

PCA minimizes total reconstruction error across all training points:

$$\min_{\mathbf{W}_r} \sum_{i=1}^{n} \|\mathbf{x}_i - \mathbf{W}_r \mathbf{W}_r^T \mathbf{x}_i\|^2$$

Using the eigendecomposition $$\mathbf{S} = \sum_{j=1}^{d} \lambda_j \mathbf{w}_j \mathbf{w}_j^T$$, the minimum is:

$$\sum_{i=1}^{n} \text{RE}(\mathbf{x}_i) = (n-1) \sum_{j=r+1}^{d} \lambda_j$$

i.e., the sum of the discarded eigenvalues, scaled by $$n-1$$.

#### The Outlier Score

For a new point $$\mathbf{x}$$, the PCA outlier score is the reconstruction error:

$$\text{Score}_{\text{PCA}}(\mathbf{x}) = \sum_{j=r+1}^{d} (\mathbf{w}_j^T \mathbf{x})^2$$

A **weighted** variant (sometimes called the *major-minor* score) uses eigenvalue weighting:

$$\text{Score}_{\text{weighted}}(\mathbf{x}) = \sum_{j=r+1}^{d} \frac{(\mathbf{w}_j^T \mathbf{x})^2}{\lambda_j}$$

This normalizes each component by its expected variance, effectively computing a **Mahalanobis distance in the minor subspace**. Dividing by $$\lambda_j$$ amplifies deviations in directions where normal data has very little variance.

### 6.5 Choosing $$r$$

The number of retained components $$r$$ is critical:

* **Too large** $$r$$: The minor subspace is tiny; reconstruction errors are small even for outliers → low sensitivity
* **Too small** $$r$$: Normal variance bleeds into the minor subspace → high false positive rate

Common heuristics:
* Retain components explaining $$\geq 95\%$$ of total variance: $$\frac{\sum_{j=1}^{r} \lambda_j}{\sum_{j=1}^{d} \lambda_j} \geq 0.95$$
* Use the "elbow" in the scree plot
* Cross-validate if labels are available

### 6.6 Limitations

* **Linearity**: PCA only captures linear correlations; nonlinear anomalies may be missed
* **Sensitivity to scaling**: Features must be standardized
* **Gaussian assumption**: The Mahalanobis-weighted score assumes Gaussian-distributed projections

---

**Industrial Example — Sensor Monitoring in Power Plants**: A power plant monitors 50+ sensor readings (temperature, pressure, vibration, flow rate). Under normal conditions, sensors are highly correlated. PCA reduces these to 5–7 principal components. When equipment degrades, the correlation structure breaks, producing large reconstruction errors — flagging the anomaly before a physical failure occurs.

In [0]:
# ============================================================
# PCA-Based Outlier Detection — Implementation
# ============================================================

# Standardize data (critical for PCA)
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X)

# Fit PCA retaining 1 component (in 2D, minor subspace has 1 dim)
pca = PCA(n_components=1)
pca.fit(X_scaled)

# Reconstruct and compute error
X_projected = pca.transform(X_scaled)
X_reconstructed = pca.inverse_transform(X_projected)
reconstruction_error = np.sum((X_scaled - X_reconstructed) ** 2, axis=1)

# Threshold: points with RE above the 90th percentile are outliers
threshold_pca = np.percentile(reconstruction_error, 90)
y_pred_pca = np.where(reconstruction_error > threshold_pca, -1, 1)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['red' if l == -1 else 'steelblue' for l in y_pred_pca]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
axes[0].set_title('PCA Outlier Detection — Predictions')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].annotate(f'Detected: {np.sum(y_pred_pca == -1)}', xy=(0.02, 0.95),
                 xycoords='axes fraction', fontsize=11, color='red')

# Plot reconstruction error distribution
axes[1].hist(reconstruction_error[y_true == 1], bins=40, alpha=0.6,
             color='steelblue', label='Normal', density=True)
axes[1].hist(reconstruction_error[y_true == -1], bins=15, alpha=0.6,
             color='red', label='Outlier', density=True)
axes[1].axvline(threshold_pca, color='black', linestyle='--', label=f'Threshold (90th %ile)')
axes[1].set_title('Reconstruction Error Distribution')
axes[1].set_xlabel('Reconstruction Error'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# Industrial Example: Sensor Monitoring in a Power Plant
# ============================================================
# Scenario: 50 sensors monitor turbine health. Under normal
# operation, readings are correlated (temp rises with RPM, etc.).
# PCA captures this structure; anomalies break it.

np.random.seed(789)
n_readings = 5000

# Correlated normal sensor data (simplified to 6 sensors)
base = np.random.randn(n_readings, 2)
normal_sensors = np.column_stack([
    base[:, 0] * 10 + 100,                    # temperature
    base[:, 0] * 5 + base[:, 1] * 2 + 50,     # pressure (correlated)
    base[:, 1] * 8 + 200,                      # flow_rate
    base[:, 0] * 3 + base[:, 1] * 6 + 80,     # vibration
    base[:, 0] * 2 + np.random.randn(n_readings) * 0.5 + 30,  # RPM
    -base[:, 0] * 4 + base[:, 1] * 3 + 60,    # coolant_level
])

# 30 anomalous readings: correlation structure broken
n_anom = 30
anom_sensors = np.column_stack([
    np.random.normal(100, 10, n_anom),
    np.random.normal(80, 15, n_anom),     # pressure too high for temp
    np.random.normal(200, 8, n_anom),
    np.random.normal(120, 20, n_anom),    # vibration spike
    np.random.normal(30, 0.5, n_anom),
    np.random.normal(30, 10, n_anom),     # coolant drop
])

X_sensor = np.vstack([normal_sensors, anom_sensors])
y_sensor = np.array([0]*n_readings + [1]*n_anom)

X_sensor_scaled = StandardScaler().fit_transform(X_sensor)
pca_sensor = PCA(n_components=3)  # retain ~95% variance
pca_sensor.fit(X_sensor_scaled)

X_rec = pca_sensor.inverse_transform(pca_sensor.transform(X_sensor_scaled))
re = np.sum((X_sensor_scaled - X_rec)**2, axis=1)

thresh = np.percentile(re, 99)
flagged_pca = re > thresh

tp = np.sum(flagged_pca & (y_sensor == 1))
print("=" * 55)
print("PCA — POWER PLANT SENSOR MONITORING")
print("=" * 55)
print(f"Total readings      : {len(X_sensor):,}")
print(f"Actual anomalies    : {y_sensor.sum()}")
print(f"Flagged by PCA      : {flagged_pca.sum()}")
print(f"True positives      : {tp}")
print(f"Explained variance  : {pca_sensor.explained_variance_ratio_.sum():.1%} (3 components)")
print(f"Precision           : {tp/max(flagged_pca.sum(),1):.2%}")
print(f"Recall              : {tp/y_sensor.sum():.2%}")

## 7. One-Class SVM

### 7.1 Intuition

One-Class SVM (Schölkopf et al., 2001) is the anomaly detection adaptation of the classic Support Vector Machine. The idea:

> **Map data to a high-dimensional feature space via a kernel, then find the smallest hypersphere (or the hyperplane with maximum margin from the origin) that encloses the normal data.**

Points outside this boundary are anomalies. The kernel trick allows learning complex, nonlinear boundaries without explicitly computing the high-dimensional mapping.

### 7.2 Problem Formulation

Given training data $$\{x_1, \ldots, x_n\}$$ (assumed mostly normal), map each point via $$\Phi: \mathbb{R}^d \to \mathcal{H}$$ (a reproducing kernel Hilbert space). Find a hyperplane that separates the mapped data from the origin with maximum margin.

**Primal Problem**:

$$\min_{\mathbf{w}, \rho, \boldsymbol{\xi}} \;\; \frac{1}{2}\|\mathbf{w}\|^2 - \rho + \frac{1}{\nu n} \sum_{i=1}^{n} \xi_i$$

$$\text{subject to} \quad \mathbf{w} \cdot \Phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0, \quad \forall i$$

where:
* $$\mathbf{w}$$ is the normal to the separating hyperplane in $$\mathcal{H}$$
* $$\rho$$ is the offset (margin from origin)
* $$\xi_i$$ are slack variables (allowing some points on the wrong side)
* $$\nu \in (0, 1]$$ is a hyperparameter: an upper bound on the fraction of outliers and a lower bound on the fraction of support vectors

In [0]:
# ============================================================
# Isolation Forest — Implementation on Synthetic Data
# ============================================================

# Train the model
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=0.1,  # ~10% outliers in our synthetic data
    random_state=42
)
y_pred_if = iso_forest.fit_predict(X)           # 1 = inlier, -1 = outlier
scores_if = iso_forest.decision_function(X)     # lower = more anomalous

# --- Visualization: Predictions + Decision Boundary ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Predicted labels
colors = ['red' if l == -1 else 'steelblue' for l in y_pred_if]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
axes[0].set_title('Isolation Forest — Predictions')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
n_detected = np.sum(y_pred_if == -1)
axes[0].annotate(f'Detected outliers: {n_detected}', xy=(0.02, 0.95),
                 xycoords='axes fraction', fontsize=11, color='red')

# Panel 2: Anomaly score heatmap
xx, yy = np.meshgrid(np.linspace(-10, 10, 200), np.linspace(-10, 10, 200))
Z = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=30, cmap='RdBu_r', alpha=0.8)
axes[1].scatter(X[:, 0], X[:, 1], c='black', s=8, alpha=0.4)
axes[1].contour(xx, yy, Z, levels=[0], colors='black', linewidths=2, linestyles='--')
axes[1].set_title('Isolation Forest — Anomaly Score Landscape')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### 7.3 Dual Formulation — Full Derivation

Using Lagrange multipliers $$\alpha_i \geq 0$$ and $$\beta_i \geq 0$$ for the constraints:

$$\mathcal{L} = \frac{1}{2}\|\mathbf{w}\|^2 - \rho + \frac{1}{\nu n}\sum_i \xi_i - \sum_i \alpha_i(\mathbf{w} \cdot \Phi(x_i) - \rho + \xi_i) - \sum_i \beta_i \xi_i$$

Setting partial derivatives to zero:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = 0 \implies \mathbf{w} = \sum_i \alpha_i \Phi(x_i)$$

$$\frac{\partial \mathcal{L}}{\partial \rho} = 0 \implies \sum_i \alpha_i = 1$$

$$\frac{\partial \mathcal{L}}{\partial \xi_i} = 0 \implies \alpha_i \leq \frac{1}{\nu n}$$

Substituting back, the **dual problem** is:

$$\min_{\boldsymbol{\alpha}} \;\; \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j K(x_i, x_j)$$

$$\text{subject to} \quad \sum_i \alpha_i = 1, \quad 0 \leq \alpha_i \leq \frac{1}{\nu n}$$

where $$K(x_i, x_j) = \Phi(x_i) \cdot \Phi(x_j)$$ is the kernel function.

**Decision function**: A new point $$x$$ is classified as:

$$f(x) = \text{sign}\left(\sum_i \alpha_i K(x_i, x) - \rho\right)$$

$$f(x) = +1$$ (inlier) or $$f(x) = -1$$ (outlier).

### 7.4 The Role of $$\nu$$

The parameter $$\nu$$ has a beautiful dual interpretation:
* **Upper bound** on the fraction of training points that are outliers (lie outside the boundary)
* **Lower bound** on the fraction of training points that are support vectors

This makes $$\nu$$ directly interpretable: setting $$\nu = 0.05$$ means at most 5% of training data can be outside the boundary.

### 7.5 Kernel Choices

| Kernel | Formula | Use Case |
| --- | --- | --- |
| RBF (Gaussian) | $$K(x, y) = \exp(-\gamma\|x-y\|^2)$$ | Most common; handles nonlinear boundaries |
| Polynomial | $$K(x, y) = (\gamma \, x \cdot y + r)^d$$ | Polynomial decision surfaces |
| Linear | $$K(x, y) = x \cdot y$$ | High-dimensional sparse data |
| Sigmoid | $$K(x, y) = \tanh(\gamma \, x \cdot y + r)$$ | Neural network analogy |

The RBF kernel is almost always the default for One-Class SVM. The $$\gamma$$ parameter controls the influence radius of each support vector.

### 7.6 Complexity

* **Training**: $$O(n^2)$$ to $$O(n^3)$$ depending on solver
* **Scoring**: $$O(n_{sv} \cdot d)$$ per point, where $$n_{sv}$$ is the number of support vectors

This makes OCSVM expensive for large datasets — prefer Isolation Forest or LOF for $$n > 50{,}000$$.

---

**Industrial Example — Manufacturing Quality Control**: A semiconductor fab measures 20+ electrical parameters per chip. One-Class SVM is trained on known-good chips; the nonlinear kernel captures complex parameter interactions. Defective chips (shorts, opens, parametric drift) fall outside the learned boundary.

In [0]:
# ============================================================
# Industrial Example: GPS Fleet Anomaly Detection with DBSCAN
# ============================================================
# Scenario: A logistics company tracks delivery trucks.
# Normal routes cluster spatially; deviations are noise.

np.random.seed(654)

# Normal route waypoints (3 common routes)
route_1 = np.random.normal([40.7128, -74.0060], [0.01, 0.01], (500, 2))
route_2 = np.random.normal([40.7580, -73.9855], [0.008, 0.008], (400, 2))
route_3 = np.random.normal([40.6892, -74.0445], [0.012, 0.012], (300, 2))

# Anomalous waypoints (unauthorized detours)
detours = np.random.uniform([40.5, -74.3], [40.9, -73.7], (20, 2))

X_gps = np.vstack([route_1, route_2, route_3, detours])
y_gps = np.array([0]*1200 + [1]*20)

db_gps = DBSCAN(eps=0.015, min_samples=10)
labels_gps = db_gps.fit_predict(X_gps)

noise_idx = np.where(labels_gps == -1)[0]
tp_gps = np.sum(y_gps[noise_idx] == 1)

print("=" * 55)
print("DBSCAN — GPS FLEET ANOMALY DETECTION")
print("=" * 55)
print(f"Total waypoints     : {len(X_gps):,}")
print(f"Actual detours      : {y_gps.sum()}")
print(f"Noise points found  : {len(noise_idx)}")
print(f"Clusters found      : {len(set(labels_gps) - {-1})}")
print(f"True positives      : {tp_gps}")
print(f"Precision           : {tp_gps/max(len(noise_idx),1):.2%}")
print(f"Recall              : {tp_gps/max(y_gps.sum(),1):.2%}")

## 9. Elliptic Envelope (Robust Covariance / Mahalanobis Distance)

### 9.1 Intuition

The Elliptic Envelope fits a **robust multivariate Gaussian** to the data, then flags points that are unlikely under this distribution. It uses the **Mahalanobis distance** — a generalization of the Z-score that accounts for correlations between features.

> **If the data is roughly elliptically distributed, the Mahalanobis distance to the center measures how anomalous each point is.**

### 9.2 Mahalanobis Distance

For a point $$\mathbf{x}$$, the Mahalanobis distance from the data center is:

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$

where $$\boldsymbol{\mu}$$ is the mean vector and $$\boldsymbol{\Sigma}$$ is the covariance matrix.

**Why not Euclidean?** Euclidean distance treats all directions equally. If Feature A has variance 100 and Feature B has variance 1, Euclidean distance is dominated by Feature A. Mahalanobis distance normalizes by the covariance, weighting all directions by their natural spread.

### 9.3 The Robustness Problem

The sample mean $$\hat{\boldsymbol{\mu}}$$ and covariance $$\hat{\boldsymbol{\Sigma}}$$ are **not robust**: outliers inflate the covariance, effectively hiding themselves (the masking effect). The Elliptic Envelope solves this using the **Minimum Covariance Determinant (MCD)** estimator.

### 9.4 MCD Estimator — Optimization

The MCD finds the subset of $$h$$ points (out of $$n$$) whose classical covariance matrix has the **smallest determinant**:

$$\min_{H \subset \{1,\ldots,n\},\; |H|=h} \det\left(\frac{1}{h} \sum_{i \in H} (\mathbf{x}_i - \bar{\mathbf{x}}_H)(\mathbf{x}_i - \bar{\mathbf{x}}_H)^T\right)$$

where $$\bar{\mathbf{x}}_H = \frac{1}{h}\sum_{i \in H} \mathbf{x}_i$$ and $$h = \lceil n(1 - \text{contamination})\rceil$$.

The MCD covariance is the **tightest ellipsoid** containing $$h$$ points. Since outliers are excluded from this subset, they don't inflate the covariance.

**Fast-MCD Algorithm** (Rousseeuw & Van Driessen, 1999): An iterative C-step algorithm that converges to a local minimum in $$O(n \cdot d^2)$$ per iteration.

### 9.5 Decision Rule

Using the robust estimates $$\hat{\boldsymbol{\mu}}_{\text{MCD}}$$ and $$\hat{\boldsymbol{\Sigma}}_{\text{MCD}}$$:

$$D_M^2(\mathbf{x}) = (\mathbf{x} - \hat{\boldsymbol{\mu}}_{\text{MCD}})^T \hat{\boldsymbol{\Sigma}}_{\text{MCD}}^{-1} (\mathbf{x} - \hat{\boldsymbol{\mu}}_{\text{MCD}})$$

Under the Gaussian assumption, $$D_M^2 \sim \chi^2_d$$. A point is flagged if:

$$D_M^2(\mathbf{x}) > \chi^2_{d,\, 1-\alpha}$$

where $$\alpha$$ is the significance level.

### 9.6 Limitations

* Assumes elliptical (roughly Gaussian) data distribution
* Not suitable for multimodal or non-convex data
* Computationally expensive for $$d > 50$$ (covariance inversion)

---

**Industrial Example — Financial Portfolio Risk**: A hedge fund monitors 15 correlated market indicators daily. The Elliptic Envelope with MCD fits a robust covariance to normal market conditions. Unusual market stress events (flash crashes, liquidity crises) show large Mahalanobis distances, triggering risk alerts.

In [0]:
# ============================================================
# One-Class SVM — Implementation on Synthetic Data
# ============================================================

X_scaled_svm = StandardScaler().fit_transform(X)

ocsvm = OneClassSVM(
    kernel='rbf',
    gamma='scale',    # gamma = 1/(n_features * X.var())
    nu=0.1            # expect ~10% outliers
)
y_pred_ocsvm = ocsvm.fit_predict(X_scaled_svm)
scores_ocsvm = ocsvm.decision_function(X_scaled_svm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['red' if l == -1 else 'steelblue' for l in y_pred_ocsvm]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
axes[0].set_title('One-Class SVM — Predictions (RBF kernel)')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].annotate(f'Detected: {np.sum(y_pred_ocsvm == -1)}', xy=(0.02, 0.95),
                 xycoords='axes fraction', fontsize=11, color='red')

Z_svm = ocsvm.decision_function(
    np.c_[xx.ravel(), yy.ravel()] / StandardScaler().fit(X).scale_
).reshape(xx.shape)

# Rescale grid for proper decision boundary
sc_temp = StandardScaler().fit(X)
xx2 = (xx - sc_temp.mean_[0]) / sc_temp.scale_[0]
yy2 = (yy - sc_temp.mean_[1]) / sc_temp.scale_[1]
Z_svm = ocsvm.decision_function(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx.shape)

axes[1].contourf(xx, yy, Z_svm, levels=30, cmap='RdBu_r', alpha=0.8)
axes[1].scatter(X[:, 0], X[:, 1], c='black', s=8, alpha=0.4)
axes[1].contour(xx, yy, Z_svm, levels=[0], colors='black', linewidths=2, linestyles='--')
axes[1].set_title('One-Class SVM — Decision Boundary')
axes[1].set_xlabel('Feature 1'); axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print(f"\nSupport vectors: {ocsvm.n_support_[0]} out of {len(X)} points")
print(f"nu = 0.1 → expected outliers ≤ {0.1*len(X):.0f}, detected = {np.sum(y_pred_ocsvm == -1)}")

In [0]:
# ============================================================
# Elliptic Envelope — Implementation on Synthetic Data
# ============================================================

ee = EllipticEnvelope(contamination=0.1, random_state=42)
y_pred_ee = ee.fit_predict(X)
scores_ee = ee.decision_function(X)
mahal_dist = ee.mahalanobis(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['red' if l == -1 else 'steelblue' for l in y_pred_ee]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
axes[0].set_title('Elliptic Envelope — Predictions')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].annotate(f'Detected: {np.sum(y_pred_ee == -1)}', xy=(0.02, 0.95),
                 xycoords='axes fraction', fontsize=11, color='red')

axes[1].hist(mahal_dist[y_true == 1], bins=40, alpha=0.6,
             color='steelblue', label='Normal', density=True)
axes[1].hist(mahal_dist[y_true == -1], bins=15, alpha=0.6,
             color='red', label='Outlier', density=True)
axes[1].set_title('Mahalanobis Distance Distribution')
axes[1].set_xlabel('Mahalanobis Distance²'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- Industrial Example: Financial Market Stress ---
np.random.seed(111)
n_days = 2000
normal_market = np.random.multivariate_normal(
    mean=[0.001, 0.002, -0.0005, 0.001],
    cov=[[0.0004, 0.0002, -0.0001, 0.0001],
         [0.0002, 0.0006, 0.0001, 0.0002],
         [-0.0001, 0.0001, 0.0003, -0.0001],
         [0.0001, 0.0002, -0.0001, 0.0005]],
    size=n_days
)

stress_events = np.array([
    [-0.08, -0.12, 0.05, -0.06],   # crash
    [-0.05, -0.09, 0.04, -0.07],   # volatility spike
    [0.10, 0.08, -0.06, 0.09],     # melt-up
    [-0.07, -0.10, 0.06, -0.05],   # flash crash
    [0.12, 0.15, -0.08, 0.11],     # bubble
])

X_mkt = np.vstack([normal_market, stress_events])
ee_mkt = EllipticEnvelope(contamination=0.003, random_state=42)
ee_mkt.fit(X_mkt)
mahal_mkt = ee_mkt.mahalanobis(X_mkt)

print("\n" + "=" * 55)
print("ELLIPTIC ENVELOPE — FINANCIAL MARKET STRESS")
print("=" * 55)
for i, day in enumerate(range(n_days, n_days+5)):
    print(f"  Stress event {i+1}: Mahalanobis² = {mahal_mkt[day]:.1f} "
          f"(threshold ≈ {np.percentile(mahal_mkt[:n_days], 99.7):.1f})")

In [0]:
# ============================================================
# Industrial Example: Semiconductor Manufacturing QC
# ============================================================
# Scenario: A chip fab measures 20 electrical parameters per wafer.
# Only "good" chips are available for training (no defect labels).
# OCSVM learns the nonlinear boundary of good chip parameter space.

np.random.seed(321)
n_good = 2000

# Good chips: correlated parameters
base_good = np.random.randn(n_good, 4)
good_chips = np.column_stack([
    base_good[:, 0] * 2 + 50,
    base_good[:, 0] * 1.5 + base_good[:, 1] * 1 + 30,
    base_good[:, 1] * 3 + base_good[:, 2] * 0.5 + 100,
    base_good[:, 2] * 2 + base_good[:, 3] * 1 + 75,
    np.sin(base_good[:, 0]) * 5 + 20 + np.random.randn(n_good) * 0.3,
])

# Defective chips (25 specimens)
n_defect = 25
defect_chips = np.column_stack([
    np.random.normal(55, 5, n_defect),
    np.random.normal(25, 8, n_defect),
    np.random.normal(110, 15, n_defect),
    np.random.normal(90, 10, n_defect),
    np.random.normal(10, 3, n_defect),
])

X_chip = np.vstack([good_chips, defect_chips])
y_chip = np.array([0]*n_good + [1]*n_defect)

X_chip_scaled = StandardScaler().fit_transform(X_chip)
ocsvm_chip = OneClassSVM(kernel='rbf', gamma='scale', nu=0.02)
ocsvm_chip.fit(X_chip_scaled[:n_good])  # train on good only (semi-supervised)
y_pred_chip = ocsvm_chip.predict(X_chip_scaled)

flagged_chip = np.where(y_pred_chip == -1)[0]
tp_chip = np.sum(y_chip[flagged_chip] == 1)

print("=" * 55)
print("ONE-CLASS SVM — SEMICONDUCTOR QC")
print("=" * 55)
print(f"Total chips         : {len(X_chip):,}")
print(f"Actual defects      : {y_chip.sum()}")
print(f"Flagged by OCSVM    : {len(flagged_chip)}")
print(f"True positives      : {tp_chip}")
print(f"Precision           : {tp_chip/max(len(flagged_chip),1):.2%}")
print(f"Recall              : {tp_chip/y_chip.sum():.2%}")

## 8. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

### 8.1 Intuition

DBSCAN (Ester et al., 1996) is primarily a clustering algorithm, but it has a built-in concept of **noise** — points that don't belong to any cluster. These noise points are natural outlier candidates.

> **Dense regions form clusters. Points in sparse regions — not reachable from any dense cluster — are noise (outliers).**

### 8.2 Core Concepts

**$$\epsilon$$-neighborhood**: The set of points within radius $$\epsilon$$ of $$x$$:

$$N_\epsilon(x) = \{y \in \mathcal{D} : d(x, y) \leq \epsilon\}$$

**Core point**: A point $$x$$ is a core point if $$|N_\epsilon(x)| \geq \text{minPts}$$.

**Border point**: Not a core point itself, but within $$\epsilon$$ of a core point.

**Noise point**: Neither core nor border — these are the **outliers**.

**Density-reachable**: Point $$y$$ is density-reachable from $$x$$ if there exists a chain of core points $$x = p_1, p_2, \ldots, p_k = y$$ where each $$p_{i+1} \in N_\epsilon(p_i)$$.

### 8.3 The Algorithm

1. For each unvisited point $$x$$: compute $$N_\epsilon(x)$$
2. If $$|N_\epsilon(x)| \geq \text{minPts}$$: start a new cluster, expand by adding all density-reachable points
3. Otherwise: mark as noise (may later become a border point if reachable from another core)

### 8.4 Parameter Selection

**$$\epsilon$$**: Use the **k-distance plot** — sort the distances to the $$k$$-th nearest neighbor. The "elbow" suggests $$\epsilon$$.

**minPts**: Rule of thumb: $$\text{minPts} \geq d + 1$$ where $$d$$ is dimensionality. Common choices: 5 for 2D data, $$2d$$ for higher dimensions.

### 8.5 Complexity

* With spatial index (KD-tree/Ball-tree): $$O(n \log n)$$
* Without: $$O(n^2)$$

### 8.6 Limitations for Outlier Detection

* Not designed primarily for outlier detection — outlier identification is a side effect
* Sensitive to $$\epsilon$$ and minPts; poor choices lead to many false noise points
* Struggles with clusters of vastly different densities (OPTICS / HDBSCAN fix this)

---

**Industrial Example — GPS Trajectory Anomalies**: Fleet management systems track vehicle GPS paths. Normal routes form dense spatial clusters. Vehicles deviating from routes (detours, unauthorized trips) create sparse trajectories that DBSCAN labels as noise.

## 10. Autoencoders for Outlier Detection

### 10.1 Intuition

An **autoencoder** is a neural network trained to reconstruct its input through a bottleneck. The network learns a compressed representation of "normal" patterns. When presented with an anomaly, the reconstruction is poor, producing a high **reconstruction error**.

> **If the network can't faithfully reconstruct a point, that point doesn't match the patterns it learned — it's an anomaly.**

This is the **nonlinear generalization of PCA**: while PCA captures linear relationships, autoencoders with nonlinear activations capture complex, hierarchical patterns.

### 10.2 Architecture

```
Input (d)  →  Encoder  →  Bottleneck (r << d)  →  Decoder  →  Output (d)
  x                         z = f_enc(x)                      x̂ = f_dec(z)
```

The encoder $$f_{\text{enc}}: \mathbb{R}^d \to \mathbb{R}^r$$ compresses; the decoder $$f_{\text{dec}}: \mathbb{R}^r \to \mathbb{R}^d$$ reconstructs.

### 10.3 Optimization (Loss Function)

The autoencoder minimizes the **mean squared reconstruction error**:

$$\mathcal{L}(\theta) = \frac{1}{n} \sum_{i=1}^{n} \|\mathbf{x}_i - f_{\text{dec}}(f_{\text{enc}}(\mathbf{x}_i; \theta_{\text{enc}}); \theta_{\text{dec}})\|^2$$

where $$\theta = \{\theta_{\text{enc}}, \theta_{\text{dec}}\}$$ are all network weights.

This is optimized via **backpropagation** and **stochastic gradient descent** (SGD / Adam).

#### Gradient Update (for Adam optimizer)

For each parameter $$\theta_j$$:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_j \leftarrow \theta_j - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

where $$g_t = \nabla_{\theta_j} \mathcal{L}$$ is the gradient.

### 10.4 Anomaly Score

$$\text{Score}_{\text{AE}}(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$$

Points with score above a threshold (e.g., 95th or 99th percentile of training reconstruction errors) are flagged as anomalies.

### 10.5 Variants

* **Variational Autoencoder (VAE)**: Adds a probabilistic latent space; anomaly score uses both reconstruction error and KL divergence
* **Denoising Autoencoder**: Trained on corrupted inputs; more robust representations
* **Sparse Autoencoder**: Adds sparsity penalty to the bottleneck
* **LSTM-Autoencoder**: For sequential/time-series anomaly detection
* **Deep SVDD**: Combines autoencoder pretraining with one-class objective — maps normal data to a hypersphere center

### 10.6 When to Use Autoencoders

* High-dimensional data (images, text embeddings, sensor arrays)
* Complex nonlinear relationships between features
* Large training datasets (neural networks are data-hungry)
* When simpler methods (PCA, IF) aren't capturing the anomaly patterns

---

**Industrial Example — Predictive Maintenance in Wind Turbines**: A wind farm collects 100+ sensor readings per turbine (vibration spectra, temperatures, power output, pitch angles). An LSTM-Autoencoder learns normal operating sequences. When a bearing begins to degrade, the subtle shift in vibration patterns increases reconstruction error days before a conventional threshold alarm.

In [0]:
# ============================================================
# Autoencoder for Outlier Detection (using PyTorch-style with sklearn)
# ============================================================
# We use sklearn's MLPRegressor as a simple autoencoder proxy.
# For production, use PyTorch/TensorFlow with proper architecture.

from sklearn.neural_network import MLPRegressor

X_scaled_ae = StandardScaler().fit_transform(X)

# Train autoencoder (MLP as identity mapping with bottleneck)
# Architecture: 2 -> 8 -> 2 -> 8 -> 2 (bottleneck = 2 is same as input
# for 2D data, so we use higher-dim example)

# For demonstration: create a higher-dim dataset
np.random.seed(42)
n_feat = 10
X_high = np.random.randn(300, n_feat) @ np.random.randn(n_feat, n_feat) * 0.5
X_high += np.random.randn(300, n_feat) * 0.1  # add noise
# Add outliers
X_high_outliers = np.random.uniform(-5, 5, (30, n_feat))
X_high_all = np.vstack([X_high, X_high_outliers])
y_high = np.array([1]*300 + [-1]*30)

X_ae_scaled = StandardScaler().fit_transform(X_high_all)

# Autoencoder: input(10) -> 64 -> 32 -> 3(bottleneck) -> 32 -> 64 -> 10
ae = MLPRegressor(
    hidden_layer_sizes=(64, 32, 3, 32, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

# Train: input = output (identity reconstruction)
ae.fit(X_ae_scaled, X_ae_scaled)

# Compute reconstruction error
X_reconstructed = ae.predict(X_ae_scaled)
re_ae = np.mean((X_ae_scaled - X_reconstructed) ** 2, axis=1)

thresh_ae = np.percentile(re_ae, 90)
y_pred_ae = np.where(re_ae > thresh_ae, -1, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(re_ae[y_high == 1], bins=40, alpha=0.6,
             color='steelblue', label='Normal', density=True)
axes[0].hist(re_ae[y_high == -1], bins=15, alpha=0.6,
             color='red', label='Outlier', density=True)
axes[0].axvline(thresh_ae, color='black', linestyle='--', label='Threshold')
axes[0].set_title('Autoencoder Reconstruction Error Distribution')
axes[0].set_xlabel('Mean Squared Error'); axes[0].set_ylabel('Density')
axes[0].legend()

# 2D PCA projection for visualization
pca_viz = PCA(n_components=2)
X_2d = pca_viz.fit_transform(X_ae_scaled)
colors_ae = ['red' if l == -1 else 'steelblue' for l in y_pred_ae]
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=colors_ae, s=20, alpha=0.6)
axes[1].set_title('Autoencoder Detections (PCA projection)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

tp_ae = np.sum((y_pred_ae == -1) & (y_high == -1))
print(f"\nAutoencoder: {np.sum(y_pred_ae == -1)} flagged, "
      f"{tp_ae}/{30} true outliers caught")

In [0]:
# ============================================================
# DBSCAN — Implementation on Synthetic Data
# ============================================================

X_scaled_db = StandardScaler().fit_transform(X)

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled_db)

# In DBSCAN: -1 = noise (outlier), >= 0 = cluster id
y_pred_db = np.where(labels_db == -1, -1, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Cluster assignments
unique_labels = set(labels_db)
colors_db = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))
for k, col in zip(sorted(unique_labels), colors_db):
    mask = labels_db == k
    if k == -1:
        axes[0].scatter(X[mask, 0], X[mask, 1], c='red', s=50,
                       marker='x', label='Noise (outlier)')
    else:
        axes[0].scatter(X[mask, 0], X[mask, 1], c=[col], s=20,
                       alpha=0.6, label=f'Cluster {k}')
axes[0].set_title('DBSCAN — Cluster Assignments')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].legend()

# Panel 2: k-distance plot for eps selection
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled_db)
dist, _ = nn.kneighbors(X_scaled_db)
k_dist = np.sort(dist[:, -1])[::-1]
axes[1].plot(k_dist)
axes[1].set_title('k-Distance Plot (k=5) — for eps selection')
axes[1].set_xlabel('Points (sorted)'); axes[1].set_ylabel('5-NN Distance')
axes[1].axhline(y=0.5, color='red', linestyle='--', label='eps=0.5')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nClusters found: {len(set(labels_db) - {-1})}")
print(f"Noise points (outliers): {np.sum(labels_db == -1)}")
print(f"True outliers caught: {np.sum((labels_db == -1) & (y_true == -1))}/{n_outliers}")

## 11. Comparative Analysis — Choosing the Right Method

### 11.1 Method Comparison

| Method | Type | Handles Non-linear | Local Sensitivity | Scalability | Interpretability | Best For |
| --- | --- | --- | --- | --- | --- | --- |
| **Z-Score / IQR** | Statistical | No | No | $$O(n)$$ | High | Quick univariate screening |
| **Isolation Forest** | Ensemble | Partially | No | $$O(n \log \psi)$$ | Medium | Large-scale, general purpose |
| **LOF** | Density | No | **Yes** | $$O(n \log n)$$ | Medium | Varying-density clusters |
| **PCA** | Linear | No | No | $$O(nd^2)$$ | High | Correlated features, sensor data |
| **One-Class SVM** | Kernel | **Yes** | No | $$O(n^2)$$ | Low | Small–medium, nonlinear boundaries |
| **DBSCAN** | Clustering | Partially | Partially | $$O(n \log n)$$ | Medium | Spatial data, natural clusters |
| **Elliptic Envelope** | Statistical | No | No | $$O(n d^2)$$ | High | Gaussian-like, moderate $$d$$ |
| **Autoencoder** | Deep Learning | **Yes** | No | $$O(n)$$ per epoch | Low | High-dim, complex patterns |

### 11.2 Decision Flowchart

```
Start
  │
  ├─ Data < 1K points?  →  LOF or Elliptic Envelope
  │
  ├─ Data > 100K points?  →  Isolation Forest (scales best)
  │
  ├─ Features highly correlated?  →  PCA or Elliptic Envelope
  │
  ├─ Varying density clusters?  →  LOF (local adaptation)
  │
  ├─ Need nonlinear boundary?  →  One-Class SVM or Autoencoder
  │
  ├─ High-dimensional (d > 50)?  →  Autoencoder or Isolation Forest
  │
  ├─ Need interpretability?  →  Z-Score / IQR / PCA / Elliptic Envelope
  │
  └─ Spatial / GPS data?  →  DBSCAN / HDBSCAN
```

### 11.3 Practical Recommendations

1. **Always start simple**: Z-Score or IQR for initial exploration; Isolation Forest as a strong baseline
2. **Ensemble approaches**: Combine 2–3 methods and take the intersection (high-confidence anomalies) or union (high-recall)
3. **Standardize features** before applying any distance or density-based method
4. **Contamination parameter**: If unknown, start with 1–5% and iterate based on domain feedback
5. **Evaluate without labels**: Use the anomaly score distribution — look for a clear separation between the bulk and the tail
6. **Time-series data**: Consider windowed features + any static method, or dedicated approaches (LSTM-AE, Spectral Residual)
7. **Production systems**: Isolation Forest for batch; streaming variants (Half-Space Trees, RRCF) for real-time

## References and Further Reading

### Foundational Papers

1. **Hawkins, D.M.** (1980). *Identification of Outliers*. Chapman and Hall.
2. **Liu, F.T., Ting, K.M., & Zhou, Z.H.** (2008). Isolation Forest. *Proc. IEEE ICDM*.
3. **Breunig, M.M., Kriegel, H.P., Ng, R.T., & Sander, J.** (2000). LOF: Identifying Density-Based Local Outliers. *Proc. ACM SIGMOD*.
4. **Schölkopf, B., Platt, J.C., Shawe-Taylor, J., Smola, A.J., & Williamson, R.C.** (2001). Estimating the Support of a High-Dimensional Distribution. *Neural Computation*.
5. **Ester, M., Kriegel, H.P., Sander, J., & Xu, X.** (1996). A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise. *Proc. KDD*.
6. **Rousseeuw, P.J., & Van Driessen, K.** (1999). A Fast Algorithm for the Minimum Covariance Determinant Estimator. *Technometrics*.
7. **Hariri, S., Kind, M.C., & Brunner, R.J.** (2019). Extended Isolation Forest. *IEEE Trans. Knowledge and Data Engineering*.

### Textbooks

8. **Aggarwal, C.C.** (2017). *Outlier Analysis* (2nd ed.). Springer.
9. **Chandola, V., Banerjee, A., & Kumar, V.** (2009). Anomaly Detection: A Survey. *ACM Computing Surveys*.
10. **Ruff, L., et al.** (2021). A Unifying Review of Deep and Shallow Anomaly Detection. *Proceedings of the IEEE*.

### Software

* **scikit-learn**: `sklearn.ensemble.IsolationForest`, `sklearn.neighbors.LocalOutlierFactor`, `sklearn.covariance.EllipticEnvelope`, `sklearn.svm.OneClassSVM`
* **PyOD** (Python Outlier Detection): Comprehensive library with 40+ algorithms — `pip install pyod`
* **Alibi Detect**: Production-grade anomaly detection with drift monitoring

---

*This notebook was designed as a self-contained reference. Each section can be studied independently. Run the code cells sequentially to reproduce all visualizations and examples.*

In [0]:
# ============================================================
# Industrial Example: Credit Card Fraud Detection
# ============================================================
# Scenario: A bank processes millions of transactions daily.
# Fraudulent transactions are rare (~0.2%) but costly.
# Isolation Forest is ideal because:
#   - No labeled fraud data is needed (unsupervised)
#   - Fraud patterns are "few and different" — the core IF assumption
#   - Scales to millions of transactions efficiently

# Simulate transaction data
np.random.seed(123)
n_transactions = 10000

# Normal transactions
normal_amount = np.random.lognormal(mean=3.5, sigma=1.0, size=n_transactions)
normal_frequency = np.random.poisson(lam=5, size=n_transactions).astype(float)
normal_distance = np.abs(np.random.normal(loc=10, scale=5, size=n_transactions))

# Fraudulent transactions (50 = 0.5%)
n_fraud = 50
fraud_amount = np.random.lognormal(mean=7.0, sigma=0.5, size=n_fraud)      # much higher amounts
fraud_frequency = np.random.poisson(lam=30, size=n_fraud).astype(float)     # rapid-fire transactions
fraud_distance = np.abs(np.random.normal(loc=500, scale=100, size=n_fraud)) # distant locations

# Combine into a DataFrame
df_transactions = pd.DataFrame({
    'amount': np.concatenate([normal_amount, fraud_amount]),
    'daily_frequency': np.concatenate([normal_frequency, fraud_frequency]),
    'distance_from_home_km': np.concatenate([normal_distance, fraud_distance]),
    'is_fraud': [0]*n_transactions + [1]*n_fraud
})

# Apply Isolation Forest
features = ['amount', 'daily_frequency', 'distance_from_home_km']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_transactions[features])

if_model = IsolationForest(n_estimators=200, contamination=0.005, random_state=42)
df_transactions['if_prediction'] = if_model.fit_predict(X_scaled)
df_transactions['if_score'] = if_model.decision_function(X_scaled)

# Evaluate
flagged = df_transactions[df_transactions['if_prediction'] == -1]
true_positives = flagged['is_fraud'].sum()
false_positives = len(flagged) - true_positives

print("=" * 55)
print("ISOLATION FOREST — CREDIT CARD FRAUD DETECTION")
print("=" * 55)
print(f"Total transactions : {len(df_transactions):,}")
print(f"Actual frauds      : {df_transactions['is_fraud'].sum()}")
print(f"Flagged by IF      : {len(flagged)}")
print(f"True positives     : {true_positives}")
print(f"False positives    : {false_positives}")
print(f"Precision          : {true_positives/len(flagged):.2%}")
print(f"Recall             : {true_positives/df_transactions['is_fraud'].sum():.2%}")

## 5. Local Outlier Factor (LOF)

### 5.1 Intuition

LOF (Breunig et al., 2000) introduces a fundamental shift: **outlierness is local, not global.**

Consider a dataset with two clusters — one dense and one sparse. A point near the sparse cluster might be farther from its neighbors than any point in the dense cluster, yet it's perfectly normal *within the context of its local neighborhood*. Global methods (Z-score, IF) might incorrectly flag it.

LOF solves this by comparing each point's density to the density of its neighbors. A point is an outlier if its local density is **significantly lower** than that of its neighbors.

### 5.2 Building Blocks

**$$k$$-distance**: The distance from $$x$$ to its $$k$$-th nearest neighbor:

$$\text{k\text{-}dist}(x) = d(x, o_k)$$

where $$o_k$$ is the $$k$$-th nearest neighbor of $$x$$ under distance metric $$d$$ (typically Euclidean).

**$$k$$-distance neighborhood**: The set of points within the $$k$$-distance:

$$N_k(x) = \{y \in \mathcal{D} \setminus \{x\} : d(x, y) \leq \text{k\text{-}dist}(x)\}$$

Note: $$|N_k(x)| \geq k$$ because of ties.

**Reachability distance**: A smoothed distance that prevents instability when points are very close:

$$\text{reach\text{-}dist}_k(x, y) = \max\{\text{k\text{-}dist}(y),\; d(x, y)\}$$

This means: from $$x$$'s perspective, $$y$$ is *at least* as far as $$y$$'s own $$k$$-distance. If $$x$$ is well inside $$y$$'s neighborhood, the reachability distance is clamped to $$\text{k\text{-}dist}(y)$$, preventing numerical instability for tightly packed points.

### 5.3 Full Mathematical Derivation

#### Local Reachability Density (LRD)

The LRD of a point $$x$$ captures the inverse of the average reachability distance from $$x$$ to its neighbors:

$$\text{lrd}_k(x) = \left( \frac{\sum_{y \in N_k(x)} \text{reach\text{-}dist}_k(x, y)}{|N_k(x)|} \right)^{-1}$$

$$= \frac{|N_k(x)|}{\sum_{y \in N_k(x)} \text{reach\text{-}dist}_k(x, y)}$$

**Interpretation**: High LRD means the point is in a **dense** region (short average reachability distances). Low LRD means the point is in a **sparse** region.

#### The LOF Score

The LOF of $$x$$ compares its density to the densities of its neighbors:

$$\text{LOF}_k(x) = \frac{1}{|N_k(x)|} \sum_{y \in N_k(x)} \frac{\text{lrd}_k(y)}{\text{lrd}_k(x)}$$

$$= \frac{\sum_{y \in N_k(x)} \text{lrd}_k(y)}{|N_k(x)| \cdot \text{lrd}_k(x)}$$

**Interpretation**:

| LOF Value | Meaning |
| --- | --- |
| $$\text{LOF} \approx 1$$ | Similar density to neighbors → **inlier** |
| $$\text{LOF} \gg 1$$ | Much lower density than neighbors → **outlier** |
| $$\text{LOF} < 1$$ | Denser than neighbors → **strong inlier** |

Typically, a threshold of $$\text{LOF} > 1.5$$ or $$\text{LOF} > 2.0$$ is used.

### 5.4 Properties and Theoretical Guarantees

**Locality**: LOF adapts to varying density across the dataset — this is its defining advantage over global methods.

**Sensitivity to $$k$$**: Small $$k$$ makes LOF sensitive to micro-structure (noisy); large $$k$$ smooths out local effects. A common practice is to evaluate LOF over a range of $$k$$ values and aggregate.

**Computational Complexity**:
* Naive: $$O(n^2)$$ for all pairwise distances
* With KD-tree or Ball-tree: $$O(n \log n)$$ on average for low-dimensional data
* High-dimensional data: degrades to $$O(n^2)$$ due to the curse of dimensionality

### 5.5 Variants

* **LOCI (Local Correlation Integral)**: Replaces the $$k$$-NN distance with a continuous kernel; automatically determines the local neighborhood radius
* **LoOP (Local Outlier Probabilities)**: Converts LOF-like scores into calibrated probabilities using a Gaussian model of local distances
* **COF (Connectivity-based Outlier Factor)**: Accounts for the chaining pattern of neighborhoods; better for low-density trails connecting clusters
* **INFLO**: Considers both $$k$$-NN and reverse $$k$$-NN to handle density transitions more gracefully

---

**Industrial Example — Network Intrusion Detection**: In a corporate network, normal traffic (HTTP, DNS) has consistent packet sizes and intervals. An intrusion (port scan, data exfiltration) shows anomalous local behavior — packets may appear normal globally but are unlike their temporal neighbors. LOF's local sensitivity catches these subtle shifts where global methods fail.

In [0]:
# ============================================================
# Local Outlier Factor — Implementation on Synthetic Data
# ============================================================

# LOF in sklearn uses novelty=False for outlier detection (training = scoring)
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.1,
    metric='euclidean'
)
y_pred_lof = lof.fit_predict(X)                   # 1 = inlier, -1 = outlier
scores_lof = lof.negative_outlier_factor_          # more negative = more anomalous

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Predictions
colors = ['red' if l == -1 else 'steelblue' for l in y_pred_lof]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.6)
axes[0].set_title('LOF — Predictions (k=20)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
n_detected = np.sum(y_pred_lof == -1)
axes[0].annotate(f'Detected outliers: {n_detected}', xy=(0.02, 0.95),
                 xycoords='axes fraction', fontsize=11, color='red')

# Panel 2: LOF score bubble plot (larger = more outlier-like)
# Normalize scores for visualization
norm_scores = (scores_lof - scores_lof.min()) / (scores_lof.max() - scores_lof.min())
scatter = axes[1].scatter(
    X[:, 0], X[:, 1],
    c=scores_lof, cmap='RdBu',
    s=200 * (1 - norm_scores) + 10,   # larger circles = more anomalous
    alpha=0.6, edgecolors='gray', linewidths=0.3
)
plt.colorbar(scatter, ax=axes[1], label='LOF Score (negative_outlier_factor_)')
axes[1].set_title('LOF — Score Visualization')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

# Show the effect of k
print("\nEffect of k (n_neighbors) on detection count:")
for k in [5, 10, 20, 50, 100]:
    lof_k = LocalOutlierFactor(n_neighbors=k, contamination=0.1)
    y_k = lof_k.fit_predict(X)
    n_out = np.sum(y_k == -1)
    n_correct = np.sum((y_k == -1) & (y_true == -1))
    print(f"  k={k:3d} → {n_out} detected, {n_correct}/{n_outliers} true outliers caught")

In [0]:
# ============================================================
# Industrial Example: Network Intrusion Detection with LOF
# ============================================================
# Scenario: A SOC (Security Operations Center) monitors network traffic.
# Normal traffic has consistent patterns; intrusions show locally
# anomalous behavior. LOF excels here because:
#   - Network traffic has varying density (web vs. DB vs. IoT subnets)
#   - A global threshold would flood analysts with false positives
#   - LOF adapts to each subnet's normal density profile

np.random.seed(456)

# Normal web traffic (high volume, small packets)
web_traffic = np.column_stack([
    np.random.normal(500, 100, 3000),      # packet_size (bytes)
    np.random.normal(100, 20, 3000),        # packets_per_second
    np.random.normal(0.05, 0.01, 3000),     # avg_latency (ms)
])

# Normal database traffic (lower volume, larger packets)
db_traffic = np.column_stack([
    np.random.normal(4000, 500, 1000),
    np.random.normal(10, 3, 1000),
    np.random.normal(0.5, 0.1, 1000),
])

# Intrusion attempts (40 instances)
# Port scan: many tiny packets, very high frequency
port_scan = np.column_stack([
    np.random.normal(60, 10, 20),
    np.random.normal(500, 50, 20),       # extremely high frequency
    np.random.normal(0.001, 0.0005, 20),
])

# Data exfiltration: huge packets, low frequency, high latency
exfiltration = np.column_stack([
    np.random.normal(15000, 2000, 20),
    np.random.normal(2, 1, 20),
    np.random.normal(2.0, 0.5, 20),
])

X_network = np.vstack([web_traffic, db_traffic, port_scan, exfiltration])
labels_network = np.array([0]*3000 + [0]*1000 + [1]*20 + [1]*20)

# Scale and apply LOF
scaler_net = StandardScaler()
X_net_scaled = scaler_net.fit_transform(X_network)

lof_net = LocalOutlierFactor(n_neighbors=30, contamination=0.01)
y_pred_net = lof_net.fit_predict(X_net_scaled)

flagged_net = np.where(y_pred_net == -1)[0]
true_intrusions = np.sum(labels_network[flagged_net] == 1)

print("=" * 55)
print("LOF — NETWORK INTRUSION DETECTION")
print("=" * 55)
print(f"Total connections   : {len(X_network):,}")
print(f"Actual intrusions   : {labels_network.sum()}")
print(f"Flagged by LOF      : {len(flagged_net)}")
print(f"True positives      : {true_intrusions}")
print(f"Precision           : {true_intrusions/max(len(flagged_net),1):.2%}")
print(f"Recall              : {true_intrusions/labels_network.sum():.2%}")